# Workshop — 1. Simulation and Data Collection

In this notebook, we build the dataset that will be used to adapt a **Vision-Language-Action model**.

The complete workshop flow is:

```text
Notebook 1                    Notebook 2                  Notebook 3
scripted teacher ─ dataset ─▶ zero-shot SmolVLA fails ─▶ fine-tuning and comparison
```

The first two notebooks intentionally use the same contract:

- robot: **SO100** in MuJoCo simulation;
- task: `Pick up the cube and place it in the box.`;
- cameras: `top` and `wrist`;
- state and action: six SO100 joints;
- metric: `placed_in_box`.

> The teacher reads the cube's ground-truth position from MuJoCo. This is a useful privilege for generating demonstrations, not a capability we attribute to the VLA.

## 1. Installation

We use the same runtime revision as Notebook 2. The critical path remains simulation-only: no hardware is initialized.

In [ ]:
%pip install -q -r requirements.txt

## 2. Environment and Imports

`MUJOCO_GL=cgl` enables off-screen rendering on macOS. It must be set before importing MuJoCo.

In [ ]:
import os
import platform

os.environ.setdefault("MUJOCO_GL", "cgl" if platform.system() == "Darwin" else "egl")
if platform.system() == "Darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

In [ ]:
import json
import sys
from pathlib import Path

from IPython.display import Video, display
from PIL import Image

sys.path.insert(0, str(Path("code").resolve()))

from vla_pick import INSTRUCTION, JOINT_KEYS, build_scene, cube_diagnostics
from so100_teacher import (
    DEMO_CUBE_POSITIONS,
    SO100PickPlaceTeacher,
    prepare_episode,
    teacher_success,
)

print("Task:", INSTRUCTION)
print("Action keys:", JOINT_KEYS)

## 3. The Same Scene as the VLA

`build_scene()` is imported from the same module used by Notebook 2. This prevents the cube, target box, cameras, and initial pose from drifting apart accidentally.

In [ ]:
sim = build_scene()
observation = sim.get_observation("so100")

display(Image.fromarray(observation["top"]).resize((480, 360)))
display(Image.fromarray(observation["wrist"]).resize((480, 360)))
print(cube_diagnostics(sim))

Each dataset frame will contain:

```text
observation.images.top ─┐
observation.images.wrist├─▶ current state + instruction ─▶ SO100 action
observation.state ──────┘
```

The scripted policy does not use the images: it computes a privileged trajectory from the cube pose. The images are still recorded because they will be the VLA inputs.

## 4. The Scripted Teacher

The teacher executes ten phases: open, approach, descend, close, lift, carry, lower into the box, release, retreat, and settle.

For each cube position, it corrects the waypoints with inverse kinematics on the actual jaw center. Actions between waypoints use cosine interpolation, avoiding discontinuities in both the servos and the dataset.

In [ ]:
teacher = SO100PickPlaceTeacher(sim)
print("Policy:", type(teacher).__name__)
print("Provider:", teacher.provider_name)
print("Uses images:", teacher.requires_images)
print("Steps:", teacher.n_steps)
print("Phases:", teacher.phase_boundaries)

### One Demonstration Before Collection

`status="success"` only means that the rollout executed. We separately verify that the cube is actually inside the target box.

In [ ]:
TEACHER_VIDEO = Path("so100_teacher.mp4").resolve()

result = sim.run_policy(
    robot_name="so100",
    policy_object=teacher,
    instruction=INSTRUCTION,
    n_steps=teacher.n_steps,
    control_frequency=30,
    fast_mode=True,
    video={"path": str(TEACHER_VIDEO), "camera": "top", "fps": 30},
)
diagnostics = cube_diagnostics(sim)
print("run_policy status:", result["status"])
print("task diagnostics:", diagnostics)

if result["status"] != "success" or not teacher_success(sim):
    raise RuntimeError("La dimostrazione non è riuscita: non registrare il dataset.")

In [ ]:
display(Video(str(TEACHER_VIDEO), embed=True, width=640))

## 5. Data Collection

The recorder produces a `LeRobotDataset`: Parquet for state/action, MP4 for both cameras, and metadata under `meta/`.

Eight episodes are enough to complete the workshop quickly and verify the pipeline. For useful fine-tuning, increase `N_EPISODES` to at least 50 and carefully expand the scene variation.

In [ ]:
DATASET_ROOT = Path("datasets/so100_sim_pickplace").resolve()
DATASET_REPO_ID = "local/so100_sim_pickplace"
DATASET_FPS = 30
N_EPISODES = 8

print(f"Dataset: {DATASET_ROOT}")
print(f"Episodes: {N_EPISODES}")
print("WARNING: overwrite=True sostituirà un dataset esistente in questo percorso.")

In [ ]:
def require_success(result, operation):
    if result.get("status") == "success":
        return result
    text = " | ".join(
        str(item.get("text"))
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )
    raise RuntimeError(f"{operation} failed: {text or result}")


require_success(
    sim.start_recording(
        repo_id=DATASET_REPO_ID,
        root=str(DATASET_ROOT),
        task=INSTRUCTION,
        fps=DATASET_FPS,
        cameras=["top", "wrist"],
        overwrite=True,
    ),
    "start_recording",
)

episode_results = []
try:
    for episode in range(N_EPISODES):
        cube_xy = DEMO_CUBE_POSITIONS[episode % len(DEMO_CUBE_POSITIONS)]
        prepare_episode(sim, cube_xy)
        teacher = SO100PickPlaceTeacher(sim)

        rollout = sim.run_policy(
            robot_name="so100",
            policy_object=teacher,
            instruction=INSTRUCTION,
            n_steps=teacher.n_steps,
            control_frequency=DATASET_FPS,
            fast_mode=True,
        )
        require_success(rollout, f"rollout episode {episode}")

        diagnostics = cube_diagnostics(sim)
        success = bool(diagnostics["placed_in_box"])
        episode_results.append({
            "episode": episode,
            "cube_xy": cube_xy,
            "final_cube": diagnostics["cube_position_m"],
            "success": success,
        })

        # Senza questo confine, tutti i rollout diventerebbero un solo episodio.
        require_success(sim.save_episode(), f"save episode {episode}")
        print(f"episode {episode:02d} | cube={cube_xy} | success={success}")
finally:
    stop = sim.stop_recording()

require_success(stop, "stop_recording")
failed = [row for row in episode_results if not row["success"]]
if failed:
    raise RuntimeError(
        f"{len(failed)} dimostrazioni fallite sono presenti nel dataset: {failed}. "
        "Non usarlo per il fine-tuning; correggi la causa e riesegui con overwrite=True."
    )

print(f"Successful demonstrations: {len(episode_results)}/{N_EPISODES}")

## 6. Dataset Verification

We verify the episode count and the contract required for fine-tuning. The expected result is: 6D state, 6D action, a `top` camera, and a `wrist` camera.

In [ ]:
verification = sim.verify_dataset_episodes(expected=N_EPISODES)
require_success(verification, "verify_dataset_episodes")
print(verification["content"][0]["text"])

info = json.loads((DATASET_ROOT / "meta" / "info.json").read_text())
features = info["features"]

expected_images = {
    "observation.images.top",
    "observation.images.wrist",
}
actual_images = {name for name in features if name.startswith("observation.images.")}

assert info["total_episodes"] == N_EPISODES
assert features["observation.state"]["shape"] == [6]
assert features["action"]["shape"] == [6]
assert actual_images == expected_images

print("\nDataset summary")
print("  robot:   ", info.get("robot_type"))
print("  episodes:", info["total_episodes"])
print("  frames:  ", info["total_frames"])
print("  fps:     ", info["fps"])
print("  state:   ", features["observation.state"]["shape"])
print("  action:  ", features["action"]["shape"])
print("  cameras: ", sorted(actual_images))

In [ ]:
top_videos = sorted(DATASET_ROOT.glob("videos/observation.images.top/**/*.mp4"))
wrist_videos = sorted(DATASET_ROOT.glob("videos/observation.images.wrist/**/*.mp4"))
if not top_videos or not wrist_videos:
    raise RuntimeError("Manca almeno uno dei due stream video nel dataset.")

print("Top camera")
display(Video(str(top_videos[0]), embed=True, width=480))
print("Wrist camera")
display(Video(str(wrist_videos[0]), embed=True, width=480))

## Result and Next Step

We now have **SO100 demonstrations compatible with the task in Notebook 2**:

- the same `top` + `wrist` images;
- the same instruction;
- the same 6D state and 6D action;
- success verified with the same `placed_in_box` metric.

In Notebook 2, we will instead run a checkpoint trained on real-world images: inference will work, but the robot will probably fail because of domain shift.

Notebook 3 will use this dataset for fine-tuning and load the new checkpoint with **radian** units, matching the values recorded by the simulator.